# 第 6 集 — `lores~`:filter 与 sample 循环

⏱ 约 30 分钟 · 难度 ★★★

**这一集控制的音乐元素:brightness(明暗)和 resonance(共鸣的 "鼻音" 感)。**

**Max 里的对应 patch:** saw → `onepole~`,然后 saw → `lores~`(或 `svf~`)。

**你会听到:**
1. 原始的亮 saw,和过了 lowpass 之后变闷的同一个音
2. 加上 resonance 后,在 cutoff 附近多出的 "鼻音 / 口哨" 尖峰
3. 一段白噪声被 filter 变成 "风声" 和 "口哨"

**学完你能做到:**
- 解释为什么 filter 必须写循环,而 oscillator 不用
- 逐行读懂一个 one-pole lowpass
- 把它和 gen~ 里的 `history` 对上
- 使用带 resonance 的 `lowpass()`
- 用 spectrum 看 filter 做了什么

这是整个系列里概念最新的一集。慢一点没关系。

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import time

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

frames = make_frames()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

saw = osc_wavetable(frames, phase, 1.0)

`saw` 是 wavetable 的最后一个 frame(32 个 harmonic),2 秒,110 Hz。这一集用它当 filter 的输入。

## 1. 为什么 filter 不一样

到目前为止,每个 sample 都可以 **独立** 算出来:`x[n]` 只取决于 `t[n]`,和别的 sample 无关。所以可以整串一起算,不用循环。

filter 不行。filter 的输出取决于 **它自己的上一个输出** — 这叫 feedback。

要算第 100 个输出,必须先知道第 99 个输出;要算第 99 个,必须先知道第 98 个…… 所以只能 **从头开始,一个一个按顺序算**。

在 Max 里,audio engine 本来就是这么工作的,你感觉不到。你在 `gen~` 里用 `history` 的时候碰到过它:`history` 就是 "记住上一个 sample 的结果"。

在 Python 里,"一个一个按顺序" 就是 `for` 循环。

## 2. `for` 循环

`for n in range(4):` 的意思是:让 `n` 依次等于 0、1、2、3,每次执行一遍下面缩进的那些行。

In [ ]:
for n in range(4):
    print("this is round", n)

循环 **外面** 定义的变量,在循环里改了之后会被 **带到下一轮**。这就是 "记住上一次" 的办法:

In [ ]:
total = 0                          # 循环外面:只执行一次
for n in range(4):
    total = total + 10             # 每一轮都在上一轮的结果上再加 10
    print("round", n, "  total is now", total)

`total` 就是一个 `history`。

## 3. 最简单的 filter:one-pole lowpass

每个 sample 只做一件事:

> **新输出 = 上一个输出 + a × (输入 − 上一个输出)**

用话说:看一下输入离我现在的位置有多远,**朝它走一小步**。`a` 决定步子多大(0 到 1)。

### 先用 10 个数走一遍

输入是一个 "台阶":前两个是 0,然后突然跳到 1。`a = 0.5`,每次走剩余距离的一半。

In [ ]:
xs = [0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
a = 0.5

prev = 0.0                                     # 上一个输出;一开始是 0
for n in range(len(xs)):
    distance = xs[n] - prev                    # 输入离我现在有多远
    prev = prev + a * distance                 # 朝它走 a 那么大的一步
    print(f"n = {n}   input = {xs[n]}   output = {prev:.3f}")

读一下输出:输入在 `n = 2` 时 **瞬间** 跳到 1。输出没有跟着跳,而是 0.5 → 0.75 → 0.875 → …… 慢慢靠近 1。

**"瞬间的跳变" 被磨成了 "平滑的上升"。** 跳变就是高频。所以这个东西磨掉了高频 — 它是一个 lowpass。

### `a` 的大小

`a` 越小,步子越小,越跟不上快速变化,磨掉的高频越多。

In [ ]:
step = np.concatenate([np.zeros(20), np.ones(180)])        # 20 个 0,然后 180 个 1

plt.figure(figsize=(9, 3))
plt.plot(step, color="gray", label="input")
for a in [0.5, 0.1, 0.02]:
    out = np.zeros(len(step))
    prev = 0.0
    for n in range(len(step)):
        prev = prev + a * (step[n] - prev)
        out[n] = prev                                      # 把这一轮的结果存进输出 array 的第 n 格
    plt.plot(out, label=f"a = {a}")
plt.legend()
plt.xlabel("sample")
plt.show()

### 包成函数:`onepole~`

用户想给的是 cutoff(Hz),不是 `a`。第一行把 cutoff 换算成 `a`:cutoff 越高,`a` 越接近 1(步子越大,放过的高频越多)。这个换算式子不用记。

In [ ]:
def onepole(x, cutoff):
    a = 1 - np.exp(-2 * np.pi * cutoff / sr)       # cutoff (Hz) → 步子大小 a (0..1)
    y = np.zeros(len(x))                           # 先准备一个空的输出 array
    prev = 0.0                                     # = gen~ 里的 history
    for n in range(len(x)):                        # n = 0, 1, 2 ... 88199
        prev = prev + a * (x[n] - prev)            # 朝输入走一小步
        y[n] = prev                                # 存下这个 sample 的输出
    return y

for cutoff in [100, 1000, 10000]:
    print(f"cutoff {cutoff:>5} Hz  ->  a = {1 - np.exp(-2 * np.pi * cutoff / sr):.3f}")

### 和 gen~ 对照

同一个 filter,在 gen~ 的 codebox 里是这样写的:

```
History prev(0);
a = 1 - exp(-twopi * in2 / samplerate);
y = prev + a * (in1 - prev);
prev = y;
out1 = y;
```

| gen~ | Python |
|---|---|
| `History prev(0);` | 循环外面的 `prev = 0.0` |
| 这段代码每个 sample 自动跑一次 | 你自己写 `for n in range(len(x)):` |
| `in1` | `x[n]` |
| `out1 = y;` | `y[n] = prev` |

**唯一的区别:gen~ 的循环是隐藏的,Python 的循环要自己写出来。**

### 听

**应该听到:** 第一个是原始的亮 saw;第二个(cutoff 300 Hz)是同一个音,但明显变闷、变远。

In [ ]:
display(play(saw))
display(play(onepole(saw, 300)))

**应该看到:** saw 的尖角被磨圆了。

In [ ]:
n = int(sr * 0.03)
plt.figure(figsize=(9, 3))
plt.plot(t[:n], saw[:n], label="saw")
plt.plot(t[:n], onepole(saw, 300)[:n], label="onepole 300 Hz")
plt.legend()
plt.xlabel("time (s)")
plt.show()

### 循环的代价

整串一起算很快,循环很慢。量一下:

In [ ]:
start = time.time()
_ = np.sin(2 * np.pi * 110 * t)                # 没有 feedback:整串一起算
print(f"whole-array sine : {1000 * (time.time() - start):.2f} ms")

start = time.time()
_ = onepole(saw, 300)                          # 有 feedback:88200 轮循环
print(f"one-pole loop    : {1000 * (time.time() - start):.2f} ms")

循环慢了几十到上百倍。对 2 秒的声音来说还是一眨眼,但这就是为什么 **能不写循环就不写循环**,只有 feedback 这种非写不可的地方才写。

## 4. Highpass:顺手得到的

lowpass 留下低频。**原始信号减去 lowpass 的结果,剩下的就是高频** — 一个 highpass。

**应该听到:** 很薄、只剩 "滋滋" 的声音,低音的 "身体" 没了。

In [ ]:
highpassed = saw - onepole(saw, 2000)
play(highpassed)

## 5. 看 filter 做了什么:spectrum

Max 里你会接一个 `spectroscope~`。这里做一个对应的函数:输入一段声音,输出 "每个频率有多响"。

里面用的 FFT 这里不展开,**先当成一个现成的 object**。第 10 集会多讲一点。

- 横轴:频率(Hz)
- 纵轴:响度(dB),0 是最响的那个频率,越往下越弱

**应该看到:** saw 是一排等间距的竖线(harmonic),缓慢地往右变矮。过了 one-pole 之后,300 Hz 以上的竖线被压低了,越往右压得越多。

In [ ]:
def spectrum(x):
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))        # 每个频率的强度
    f = np.fft.rfftfreq(len(x), 1 / sr)                    # 对应的频率刻度(Hz)
    return f, 20 * np.log10(X / np.max(X) + 1e-9)          # 换成 dB,最响的是 0

def spec_plot(signals, max_freq=5000):
    plt.figure(figsize=(9, 4))
    for name in signals:
        f, db = spectrum(signals[name])
        plt.plot(f, db, label=name, linewidth=0.8)
    plt.xlim(0, max_freq)
    plt.ylim(-90, 5)
    plt.xlabel("frequency (Hz)")
    plt.ylabel("level (dB)")
    plt.legend()
    plt.show()

spec_plot({"raw saw": saw, "onepole 300 Hz": onepole(saw, 300)})

## 6. 带 resonance 的 filter:`lores~` / `svf~`

one-pole 很温和(每高一个八度只降 6 dB),也没有 resonance。synth 里常用的是 2-pole 的 **state variable filter**(`svf~`):降得更陡(12 dB / 八度),而且可以在 cutoff 处顶起一个共鸣峰。

它的结构和 one-pole **完全一样**:一个循环,循环里记住上一步的状态。区别只是:

- 状态从 1 个变成 2 个(`ic1`、`ic2`)
- 每一步的算式从 1 行变成 5 行

你在 Max 里不会打开 `lores~` 去看里面。这里也一样:**知道它是 "一个循环 + 两个 history" 就够了**,算式本身不用懂。函数里每一段都有注释。

- `cutoff`:Hz(可以是一个数,也可以是 array — 下一集会用到)
- `res`:0 到 1,越大共鸣峰越尖

In [ ]:
def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff 给一个数或一个 array 都行:统一变成和声音一样长的 array
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- 把 cutoff / res 换算成 filter 内部用的系数(每个 sample 一组)---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping:res 越大 k 越小,共鸣越强
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() 把 array 变成普通 list:在循环里逐个取数时快很多
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # 两个状态变量 = gen~ 里的两个 history

    for n in range(n_samples):                 # 一个 sample 一个 sample 按顺序算
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 就是 lowpass 的输出
        ic1 = 2 * v1 - ic1                     # 更新状态,留给下一个 sample 用
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

### 听:cutoff

**应该听到:** 三个音,越来越亮 — 200 Hz 很闷,800 Hz 中等,4000 Hz 接近原始 saw。

In [ ]:
for cutoff in [200, 800, 4000]:
    print("cutoff", cutoff, "Hz")
    display(play(lowpass(saw, cutoff)))

### 听:resonance

cutoff 固定在 800 Hz,只改 `res`。

**应该听到:** `res = 0` 是普通的闷音。`res` 升高后,在 800 Hz 附近多出一个越来越明显的 "鼻音 / 口哨" 尖峰,像嘴巴做出 "诶" 的口型。

(resonance 高的时候峰值会变大,所以后两个的 gain 调低了。)

In [ ]:
display(play(lowpass(saw, 800, res=0.0)))
display(play(lowpass(saw, 800, res=0.6), gain=0.2))
display(play(lowpass(saw, 800, res=0.9), gain=0.15))

**应该看到:** 三条线在 800 Hz 之前差不多;`res` 越大,800 Hz 那里顶起的峰越高。

In [ ]:
spec_plot({
    "res 0.0": lowpass(saw, 800, 0.0),
    "res 0.6": lowpass(saw, 800, 0.6),
    "res 0.9": lowpass(saw, 800, 0.9),
}, max_freq=3000)

## 7. Filter 一段噪声

`rng.uniform(-1, 1, n)`:`n` 个 -1 到 1 之间的随机数。每个 sample 都和前后无关 — 这就是白噪声,Max 里的 `noise~`。它包含所有频率。

**应该听到:**
1. 原始白噪声:"沙——"
2. lowpass 400 Hz:闷闷的 "轰",像远处的风或海浪
3. lowpass 1200 Hz + 很高的 resonance:噪声里浮出一个有音高的 "口哨"(在 1200 Hz 附近)

In [ ]:
rng = np.random.default_rng(0)
noise = rng.uniform(-1, 1, len(t))

display(play(noise, gain=0.15))
display(play(lowpass(noise, 400), gain=0.5))
display(play(lowpass(noise, 1200, res=0.98), gain=0.15))

## 8. 串联两个 filter

把 filter 的输出再送进一个同样的 filter,降得就陡一倍(24 dB / 八度)。Max 里就是两个 `lores~` 串起来。

**应该听到:** 第二个比第一个更闷、更 "圆"。

In [ ]:
once = lowpass(saw, 600)
twice = lowpass(once, 600)

display(play(once))
display(play(twice))

spec_plot({"raw saw": saw, "one filter": once, "two filters": twice}, max_freq=4000)

## 常见错误

| 现象 | 原因 |
|---|---|
| filter 的输出全是 0 | 忘了 `y[n] = prev`,或者把它写到了循环外面(缩进错了) |
| 输出只是变小声了,没有变闷 | `prev = 0.0` 写在了循环 **里面**,每一轮都被重置成 0,等于没有 "记住上一次" |
| 运行特别慢 | 循环里用了不必要的 numpy 运算;或者声音太长 |
| resonance 很高时爆音 | 峰值超过 1 了,调低 gain |

缩进在 Python 里是有意义的:缩进的行属于循环,不缩进的行在循环结束后才执行。

## 练习

**1.** 把 `onepole` 的 cutoff 设成 50、500、5000,听三个结果。**先猜:** 哪个最接近原始 saw?

<details><summary>答案</summary>

```python
for c in [50, 500, 5000]:
    display(play(onepole(saw, c)))
```
5000 最接近。cutoff 越高,放过的越多。
</details>

**2.** 用 `saw - lowpass(saw, 1500, res=0.5)` 做一个带 resonance 的 highpass,听一下并画 spectrum。

<details><summary>答案</summary>

```python
hp = saw - lowpass(saw, 1500, res=0.5)
play(hp, gain=0.2)
spec_plot({"highpass": hp})
```
</details>

**3.** 把第 7 节的 "口哨" 的 cutoff 改成 300 和 3000。**应该听到:** 口哨的音高跟着 cutoff 走。

<details><summary>答案</summary>

```python
for c in [300, 1200, 3000]:
    display(play(lowpass(noise, c, res=0.98), gain=0.15))
```
</details>

## 小结

| Max | Python |
|---|---|
| audio engine 逐 sample 运行 | `for n in range(len(x)):` |
| `history`(gen~) | 循环外面定义、循环里更新的变量 |
| `onepole~` | `onepole(x, cutoff)` |
| `lores~` / `svf~` | `lowpass(x, cutoff, res)` |
| `noise~` | `rng.uniform(-1, 1, n)` |
| `spectroscope~` | `spec_plot({...})` |

**规则:** 没有 feedback → 整串 array 一起算。有 feedback → 必须写循环。

**下一集:** 把 envelope 和 LFO 接到 cutoff 和 pitch 上 — modulation。